In [1]:
import cv2
from google.colab.patches import cv2_imshow #collab crashuje przy samym cv2.imshow
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

wciagniecie wszystkich zdjec  

In [2]:
zdjecia = []
for i in range(1,9):
  zdjecie = cv2.imread(f"/content/drive/MyDrive/Colab Notebooks/WMA/CoinDetection/tray{i}.jpg",cv2.IMREAD_GRAYSCALE)
  if zdjecie is None:
    print(f"zdjecie {i} nie istnieje")
    continue
  zdjecia.append(zdjecie)

for zdjecie in zdjecia:
  cv2_imshow(zdjecie)

Output hidden; open in https://colab.research.google.com to view.

Odszumienie


In [3]:
zdjeciaOdszumienie = []
temp=0

for zdjecie in zdjecia:
  sobelx = cv2.Sobel(zdjecie, cv2.CV_64F, 1, 0, ksize=5)
  sobely = cv2.Sobel(zdjecie, cv2.CV_64F, 0, 1, ksize=5)
  laplacian =np.uint8(np.absolute( cv2.Laplacian(zdjecie, cv2.CV_64F, ksize=3)))
  zdjeciaOdszumienie.append(laplacian)
  if temp == 0:
    cv2_imshow(sobelx)
    cv2_imshow(sobely)
    cv2_imshow(laplacian)
    temp+=1

Output hidden; open in https://colab.research.google.com to view.

Krawedzie + wyostrzenie + krawedzie tacki

In [4]:
def skrajneLinieKonturu(linie):
  if len(linie) == 0:
    return None, None

  linie_sorted = sorted(linie, key=lambda x: x[0])#odleglosc linii od poczatku ukladu (po rho)
  return linie_sorted[0], linie_sorted[-1] #najblizsza i najdalsza linia

def przeciecieLinii(linia1, linia2):
  rho1, theta1 = linia1
  rho2, theta2 = linia2

  A = np.array([#uklad znajdujacy przeciecie linii w formie hougha
      [np.cos(theta1), np.sin(theta1)],
      [np.cos(theta2), np.sin(theta2)]
  ])
  b = np.array([rho1, rho2])

  if np.linalg.det(A) == 0: #sprawdzenie czy nie sa rownolegle
    return None

  x0, y0 = np.linalg.solve(A,b)# przeciecie
  return int(x0), int(y0)

def grupowanieLinii(linie): #laczenie linii w grupy bo kontur robil sie tylko np po prawej
  if len(linie) == 0:
    return []

  linie_sorted = sorted(linie, key=lambda x: x[0]) #sortowanie po rho
  linie_grupowane = [[linie_sorted[0]]]

  for linia in linie_sorted[1:]:
    if abs(linia[0] - linie_grupowane[-1][-1][0])<30:#granica 30 pikseli odleglosci
      linie_grupowane[-1].append(linia)
    else:
      linie_grupowane.append([linia])

  return linie_grupowane #linie blisko siebie sa zgrupowane

def usrednionaLinia(grupa): #wybor najlepszej linii z kazdej grupy (gora dol lewo prawo)
  rho = np.mean([linia[0] for linia in grupa]) #srednia arytmetyczna z odleglosci od poczatku ukladu wspolrzednych dla grupy
  theta = np.mean([linia[1] for linia in grupa]) #srednia arytmetyczna
  return (rho, theta)

In [5]:
konturyTacki = []
zdjeciaKrawedzieTacki = []
zdjeciaLinie = []

for i in range(len(zdjecia)):
  zdjecie = np.uint8(np.absolute(zdjecia[i])).copy()

  blur = cv2.GaussianBlur(zdjecie,(5,5),0)

  canny = cv2.Canny(
      image=blur,
      threshold1=75,
      threshold2=120,
      apertureSize=3,
      L2gradient=False
  )

  zdjeciaKrawedzieTacki.append(canny)

  #houghLines
  linie = cv2.HoughLines(
      image=canny,
      rho=1.0,
      theta=float(np.pi /180),
      threshold=75
  )

  liniePionowe = []
  liniePoziome = []

  if linie is not None:
    for linia in linie:
      rho, theta = linia[0]

      #poziome
      if abs(theta) < np.pi/6 or abs(theta - np.pi) < np.pi/6:
        liniePoziome.append((rho,theta))

      elif abs(theta - np.pi/2) < np.pi/6:
        liniePionowe.append((rho,theta))

  #grupowanie
  grupyPionowe = grupowanieLinii(liniePionowe)
  grupyPoziome = grupowanieLinii(liniePoziome)

  #sortowanie i znalezienie tylko najwiekszych grup linii
  grupyPionowe = sorted(grupyPionowe, key=len, reverse=True)[:2]
  grupyPoziome = sorted(grupyPoziome, key=len, reverse=True)[:2]

  if len(grupyPionowe) == 2 or len(grupyPoziome) == 2:
    left = usrednionaLinia(grupyPionowe[0])
    right = usrednionaLinia(grupyPionowe[1])

    top = usrednionaLinia(grupyPoziome[0])
    bottom = usrednionaLinia(grupyPoziome[1])

    if left[0] > right[0]:
      left, right = right, left
    if top[0] > bottom[0]:
      top, bottom = bottom, top

  #wyznaczenie rogow na przecieciu linii
  if None not in (left,right,top,bottom):
    p1 = przeciecieLinii(left,top)
    p2 = przeciecieLinii(right,top)
    p3 = przeciecieLinii(left,bottom)
    p4 = przeciecieLinii(right,bottom)

    if None not in (p1,p2,p3,p4):
      kontur = np.array([p1,p2,p4,p3], dtype=np.int32)
      konturyTacki.append(kontur)
      cv2.polylines(zdjecie, [kontur], isClosed=True, color=(255,0,0),thickness=3)

  cv2_imshow(canny)
  cv2_imshow(zdjecie)

Output hidden; open in https://colab.research.google.com to view.

Wykrywanie kol i rozmiarow (nominalow)

In [6]:
KolaNaObraz = []

for i in range(len(zdjeciaKrawedzieTacki)):
  zdjecie = zdjecia[i].copy()
  kolorowe = cv2.cvtColor(zdjecia[i], cv2.COLOR_GRAY2BGR)

  #wykrywanie kol
  blur = cv2.GaussianBlur(zdjecie,(5,5),0)
  kolo = cv2.HoughCircles(
      blur,
      method=cv2.HOUGH_GRADIENT,
      dp=1,
      minDist=20,
      param1=50,
      param2=40,
      minRadius=20,
      maxRadius=40
  )
  kolaInfoTemp = []

  if kolo is not None:
    koloZaokraglone = np.uint16(np.around(kolo[0]))
    for(x,y,r) in koloZaokraglone:
      cv2.circle(kolorowe,(x,y),r,(0,255,0),2)
      cv2.circle(kolorowe,(x,y),1,(0,0,255),3)

      kolaInfoTemp.append(((x,y),r))

  kolaInfo = []
  kolaInfoTemp_Sorted = sorted(kolaInfoTemp, key=lambda x: x[1], reverse=True)
  max1 = kolaInfoTemp_Sorted[0][1]
  max2 = kolaInfoTemp_Sorted[1][1]

  for j in range(len(kolaInfoTemp)):
    wartosc = 0.05
    if kolaInfoTemp[j][1] == max1 or kolaInfoTemp[j][1] == max2:
      wartosc = 5

    kolaInfo.append((kolaInfoTemp[j],wartosc))
    print(kolaInfoTemp[j][1], wartosc)


  KolaNaObraz.append(kolaInfo)
  # if i == 0:
  cv2_imshow(kolorowe)

print(KolaNaObraz)

Output hidden; open in https://colab.research.google.com to view.

okreslenie zakresu tacki (6)

In [7]:
tackaZakres = []
kolaWTacce = []

for i in range(len(zdjecia)):
  zdjecie = zdjecia[i].copy()
  zdjecieKolorowe = cv2.cvtColor(zdjecie, cv2.COLOR_GRAY2BGR)

  if i >= len(konturyTacki):
    continue

  konturTacki = konturyTacki[i]

  wTacce = 0
  pozaTacka = 0

  for((x,y),r),wartosc in KolaNaObraz[i]:
    value = cv2.pointPolygonTest(konturTacki, (int(x),int(y)), measureDist=False)

    if value > 0:#na tacce
      kolor = (0,255,0)
      wTacce = round(wTacce + wartosc,2)
    elif value < 0:#poza tacka
      kolor = (0,0,255)
      pozaTacka = round(pozaTacka + wartosc,2)
    else:
      kolor = (255,0,0)

    cv2.circle(zdjecieKolorowe,(int(x),int(y)),r,kolor,2)

  #znalezienie najwiekszej monety dla skali tacki
  najwieksza = max(KolaNaObraz[i], key=lambda x: x[0][1])#sortowanie po promieniu
  cmNaPiksele = 2.4 / (2* najwieksza[0][1])
  poleTackiWPikselach = cv2.contourArea(konturTacki)

  print(f"wartosc monet w tacce:{wTacce}, poza tacka:{pozaTacka}. pole tacki = {poleTackiWPikselach} pikseli czyli {round(poleTackiWPikselach  * (cmNaPiksele ** 2),2)} cm^2")
  cv2_imshow(zdjecieKolorowe)


Output hidden; open in https://colab.research.google.com to view.